# EDA and data cleaning for the PPR csv data

## Imports

In [1]:
import pandas as pd
from pathlib import Path
import re
from deepparse.parser import AddressParser
import numpy as np

/workspaces/ireland-property-analysis/.venv/lib/python3.12/site-packages/pymagnitudelight/framework/repoze/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


## read raw data

In [2]:
# Path to the file we downloaded in Iteration 1
raw_data_path = Path("../data/raw/PPR-ALL.csv")

# Note: The PPR file uses 'latin-1' or 'cp1252' encoding, not standard 'utf-8'
raw_df = pd.read_csv(raw_data_path, encoding='cp1252')

/tmp/ipykernel_9915/2864090230.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(raw_data_path, encoding='cp1252')


## read a sample from the raw data

In [3]:
print(f"Dataset contains {len(raw_df):,} rows.")
raw_df.head()

Dataset contains 768,790 rows.


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"€343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"€185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"€438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,"€400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,"€160,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


## create deep copy of the raw data

In [4]:
df = raw_df.copy(deep=True)

## clean columns names

In [5]:
df.columns = (
    df.columns
    .str.replace(r'[^\w\s]', '', regex=True) # Removes (€) and other symbols
    .str.strip()                             # Removes leading/trailing spaces
    .str.replace(' ', '_')                   # Replaces middle spaces with underscores
    .str.lower()                             # Makes everything lowercase
)
print(df.columns)

Index(['date_of_sale_ddmmyyyy', 'address', 'county', 'eircode', 'price',
       'not_full_market_price', 'vat_exclusive', 'description_of_property',
       'property_size_description'],
      dtype='object')


## Price cleaning and enrich

### clean price column from non numeric symbols

#### create a function for cleaning the price string

In [6]:
def clean_currency(price_str):
    if pd.isna(price_str):
        return None
    # remove everything that isn't a digit or descimal point
    clean_str = re.sub(r'[^\d.]','',str(price_str))
    # convert clean_str to floot which handles the .00 then to int
    try:
        return int(float(clean_str))
    except ValueError:
        return None

#### use the function to clean

In [7]:
df['price_clean'] = df['price'].apply(clean_currency)
print(f"Average Price in euros: {df['price_clean'].mean()}")

Average Price in euros: 313460.9311723618


### create boolean column for vat exclusive

In [8]:
df['is_vat_exclusive'] = df['vat_exclusive'].str.contains('Yes',case=False, na=False)

## create a smaller sample dataset for fast debugging and development

#### new folder so raw data are safe

In [9]:
# Define your paths
processed_dir = Path("../data/processed")

# Create the folder if it doesn't exist
processed_dir.mkdir(parents=True, exist_ok=True)

#### filter county dublin only

In [10]:
# 1. Filter for Dublin (Case-insensitive just in case)
dublin_df = df[df['county'].str.contains('Dublin', case=False, na=False)].copy()

# 2. Save to new folder
output_path = processed_dir / "dublin_sample.csv"
dublin_df.to_csv(output_path, index=False)

print(f"Success! Saved {len(dublin_df):,} rows to {output_path}")

Success! Saved 240,930 rows to ../data/processed/dublin_sample.csv


## Deepparse to parse address

### use deepparse on the dublin sample dataset

In [11]:
df_dublin = pd.read_csv("../data/processed/dublin_sample.csv")
#Initialize the parser (the first time takes a moment to download the model)
# We use 'fasttext' because it's lightweight and runs great on a Mac
address_parser = AddressParser(model_type="bpemb", device="cpu")
#Test on a single messy Dublin address
test_address = df_dublin['address'].iloc[0]
parsed = address_parser(test_address)

print(f"Original: {test_address}")
print(f"Parsed: {parsed}")


/tmp/ipykernel_9915/1669742642.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_dublin = pd.read_csv("../data/processed/dublin_sample.csv")
/workspaces/ireland-property-analysis/.venv/lib/python3.12/site-packages/deepparse/download_tools.py:92: UserWarning: The offline parameter is set to False, so if a new pre-trained `bpemb` model is available it will automatically be downloaded.
  warnings.warn(


Loading the embeddings model
Original: 5 Braemor Drive, Churchtown, Co.Dublin
Parsed: The unparsed address is '5 Braemor Drive, Churchtown, Co.Dublin' and the parsed address is '('5', 'StreetNumber') ('braemor', 'StreetName') ('drive', 'StreetName') ('churchtown', 'StreetName') ('co.dublin', 'Municipality')'


### function to convert address string to dictionary with address components

In [12]:
def parse_address_to_cols(address_str):
    try:
        return address_parser(address_str)
    except Exception as e:
        return {"error": str(e)}


In [13]:
def fast_parse(address_list):
    # num_workers=2 or 4 uses multiple CPU cores
    # batch_size=256 processes many addresses at once
    parsed_results = address_parser(address_list, batch_size=256)
    return [obj.to_dict() for obj in parsed_results]

In [14]:
df_to_parse = df_dublin
# 2. Split into batches
batches = np.array_split(df_to_parse['address'].tolist(), 30)
all_results = []
print(f"Starting parsing of {len(df_to_parse)} rows...")

for i, batch in enumerate(batches):
    print(f"Processing batch {i+1}/30...")
    all_results.extend(fast_parse(batch.tolist()))

# 3. Create the final DataFrame
df_parsed = pd.DataFrame(all_results)
df_final = pd.concat([df_to_parse.reset_index(drop=True), df_parsed], axis=1)

print("Done! Here is a sample:")
print(df_final[['Address', 'StreetNumber', 'StreetName']].head())

Starting parsing of 240930 rows...
Processing batch 1/30...
Vectorizing the address
Processing batch 2/30...
Vectorizing the address
Processing batch 3/30...
Vectorizing the address
Processing batch 4/30...
Vectorizing the address
Processing batch 5/30...
Vectorizing the address
Processing batch 6/30...
Vectorizing the address
Processing batch 7/30...
Vectorizing the address
Processing batch 8/30...
Vectorizing the address
Processing batch 9/30...
Vectorizing the address
Processing batch 10/30...
Vectorizing the address
Processing batch 11/30...
Vectorizing the address
Processing batch 12/30...
Vectorizing the address
Processing batch 13/30...
Vectorizing the address
Processing batch 14/30...
Vectorizing the address
Processing batch 15/30...
Vectorizing the address
Processing batch 16/30...
Vectorizing the address
Processing batch 17/30...
Vectorizing the address
Processing batch 18/30...
Vectorizing the address
Processing batch 19/30...
Vectorizing the address
Processing batch 20/30..

KeyError: "['Address'] not in index"

In [16]:
print("Done! Here is a sample:")
print(df_final[['address', 'StreetNumber', 'StreetName']].head())

Done! Here is a sample:
                                     address StreetNumber  \
0     5 Braemor Drive, Churchtown, Co.Dublin            5   
1        1 Meadow Avenue, Dundrum, Dublin 14            1   
2             12 Sallymount Avenue, Ranelagh           12   
3  206 Philipsburgh Avenue, Marino, Dublin 3          206   
4     22 Laverna Way, Castleknock, Dublin 15           22   

                   StreetName  
0    braemor drive churchtown  
1       meadow avenue dundrum  
2           sallymount avenue  
3  philipsburgh avenue marino  
4     laverna way castleknock  


In [15]:
# Save to a compressed CSV to save space, or a standard CSV
df_final.to_csv("../data/processed/ppr_dublin_parsed.csv", index=False)
print("File saved successfully!")

File saved successfully!


In [17]:
#Check for common Irish address components
# See how many times it found a 'Municipality' (usually the County/Town)
print(df_final['Municipality'].value_counts().head(10))

# Check for rows where it couldn't find a StreetName
null_streets = df_final['StreetName'].isnull().sum()
print(f"Rows without a Street Name: {null_streets}")

Municipality
dublin               16540
swords                3295
co dublin             2316
lucan                 2198
lucan dublin          1991
malahide              1805
laoghaire             1501
tallaght              1345
balbriggan dublin     1238
balbriggan            1120
Name: count, dtype: int64
Rows without a Street Name: 383


In [18]:
# Convert to lowercase, remove 'co ', and strip 'dublin' from the end 
# so 'lucan dublin' becomes just 'lucan'
df_final['Municipality_Clean'] = (
    df_final['Municipality']
    .str.lower()
    .str.replace('co ', '', regex=False)
    .str.replace(' dublin', '', regex=False)
    .str.strip()
)

print(df_final['Municipality_Clean'].value_counts().head(10))

Municipality_Clean
dublin        18901
swords         4354
lucan          4197
balbriggan     2380
malahide       2303
blackrock      1870
laoghaire      1786
tallaght       1465
donabate       1160
stillorgan     1068
Name: count, dtype: int64


In [ ]:
# See what these 383 rows actually look like
missing_streets = df_final[df_final['StreetName'].isnull()]
print(missing_streets['address'].head(10))

714                       APT17 BLOCKC, THORNBERRY SQUARE
1217                                Van Neis, Rathfarnham
1530          1-12 Chestnut Lodge, Farmleigh, Castleknock
2252                                 Blacklands, Skerries
2450                            Knightstown, Ballyboughal
2581                      25Earlsfort Road, Lucan, Dublin
2864                             Bettyville, Ballyboughal
3578    Apartment 12  Isoldes Tower, Essex Quay  Templ...
4020                        Brannock, Milverton, Skerries
5207    Apt. 1- 11 Stillorgan Haven, 405 to 407 Stillo...
Name: address, dtype: object


: 